# 03 优化器与学习率调度

本 Notebook 覆盖：
- SGD / Adam / AdamW 收敛速度对比
- 常用学习率调度器可视化
- 梯度裁剪实战

**对应文档**: `docs/03-training-advanced/01-optimizers.md`, `02-lr-schedulers.md`

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device(
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print(f'使用设备: {device}')

## 1. 优化器收敛速度对比

In [ ]:
torch.manual_seed(42)
X = torch.randn(600, 20)
y = (X[:, :5].sum(dim=1) > 0).long()
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True, num_workers=0)

def make_model():
    return nn.Sequential(
        nn.Linear(20, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 2),
    ).to(device)

def train_with_optimizer(opt_name, epochs=40):
    model = make_model()
    criterion = nn.CrossEntropyLoss()
    opts = {
        'SGD':   torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4),
        'Adam':  torch.optim.Adam(model.parameters(), lr=1e-3),
        'AdamW': torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01),
    }
    opt = opts[opt_name]
    history = []
    for _ in range(epochs):
        total = 0.0
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item()
        history.append(total / len(loader))
    return history

results = {name: train_with_optimizer(name) for name in ['SGD', 'Adam', 'AdamW']}

plt.figure(figsize=(8, 4))
for name, hist in results.items():
    plt.plot(hist, label=name)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('优化器收敛速度对比')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

for name, hist in results.items():
    print(f'{name:6s}: 最终 loss = {hist[-1]:.4f}')

## 2. 学习率调度器可视化

In [ ]:
def get_lr_history(scheduler_name, epochs=50, steps_per_epoch=20):
    model = make_model()
    opt   = torch.optim.SGD(model.parameters(), lr=0.1)
    total_steps = epochs * steps_per_epoch

    schedulers = {
        'StepLR':           torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5),
        'CosineAnnealing':  torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs),
        'OneCycleLR':       torch.optim.lr_scheduler.OneCycleLR(
                                opt, max_lr=0.1, total_steps=total_steps),
        'ExponentialLR':    torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.95),
    }
    sched = schedulers[scheduler_name]

    lrs = []
    is_step_based = scheduler_name == 'OneCycleLR'

    for epoch in range(epochs):
        if is_step_based:
            for _ in range(steps_per_epoch):
                lrs.append(opt.param_groups[0]['lr'])
                sched.step()
        else:
            lrs.append(opt.param_groups[0]['lr'])
            sched.step()
    return lrs

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
scheduler_names = ['StepLR', 'CosineAnnealing', 'OneCycleLR', 'ExponentialLR']

for ax, name in zip(axes, scheduler_names):
    lrs = get_lr_history(name)
    ax.plot(lrs)
    ax.set_title(name)
    ax.set_xlabel('Step' if name == 'OneCycleLR' else 'Epoch')
    ax.set_ylabel('Learning Rate')
    ax.grid(True, alpha=0.3)

plt.suptitle('学习率调度器对比', fontsize=14)
plt.tight_layout()
plt.show()

## 3. 梯度裁剪对比实验

In [ ]:
def get_grad_norms(use_clip: bool, max_norm: float = 1.0, epochs: int = 20):
    model     = make_model()
    criterion = nn.CrossEntropyLoss()
    opt       = torch.optim.SGD(model.parameters(), lr=0.5)  # 故意用大学习率触发梯度爆炸
    norms = []

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            norm = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None) ** 0.5
            norms.append(norm)
            if use_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            opt.step()
    return norms

norms_no_clip = get_grad_norms(use_clip=False)
norms_clip    = get_grad_norms(use_clip=True)

plt.figure(figsize=(10, 4))
plt.plot(norms_no_clip[:200], label='无梯度裁剪', alpha=0.7)
plt.plot(norms_clip[:200],    label='有梯度裁剪 (max=1.0)', alpha=0.7)
plt.xlabel('Step'); plt.ylabel('梯度 L2 范数')
plt.title('梯度裁剪效果对比')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 练习题

1. **简单**: 将 `OneCycleLR` 的 `max_lr` 改为 `0.001` 和 `1.0`，分别观察学习率曲线和训练效果
2. **中等**: 实现 Warmup + CosineDecay 的组合调度（前 10 个 epoch 线性预热，之后余弦衰减）
3. **挑战**: 在 MNIST 上对比 Adam vs AdamW，统计测试集准确率和过拟合程度